In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("PySpark Test").getOrCreate()

In [3]:
test_df = spark.read.csv("Test1.csv", header=True, inferSchema=True)
test_df.show()

+---------+----------+--------+----+----------+-----------+------+
|firstName|middleName|lastName| Age|Experience| Department|Salary|
+---------+----------+--------+----+----------+-----------+------+
|     John|      NULL|   Smith|  28|         3|      Sales| 50000|
|    Maria|    Isabel|  Cortez|NULL|         5|Engineering| 72000|
|    Chris|      NULL|   Evans|  35|      NULL|  Marketing| 60000|
|     Anna|     Marie|    NULL|  30|         2|      Sales|  NULL|
|     NULL|      NULL|   Lopez|  40|        10|         HR| 80000|
|    James|    Andrew|   Brown|NULL|         3|       NULL| 55000|
|    Sarah|      NULL|  Miller|  27|         1|Engineering| 65000|
+---------+----------+--------+----+----------+-----------+------+



In [7]:
df_filtered = test_df.select("firstName", "lastName", "Salary").limit(3)
df_filtered.show()

+---------+--------+------+
|firstName|lastName|Salary|
+---------+--------+------+
|     John|   Smith| 50000|
|    Maria|  Cortez| 72000|
|    Chris|   Evans| 60000|
+---------+--------+------+



In [ ]:

from pyspark.sql import functions as F
df_concat = test_df.withColumn("FullName", F.concat_ws(" ", test_df["firstName"], test_df["middleName"], test_df["lastName"]))
df_concat.show()

+---------+----------+--------+----+----------+-----------+------+-------------------+
|firstName|middleName|lastName| Age|Experience| Department|Salary|           FullName|
+---------+----------+--------+----+----------+-----------+------+-------------------+
|     John|      NULL|   Smith|  28|         3|      Sales| 50000|         John Smith|
|    Maria|    Isabel|  Cortez|NULL|         5|Engineering| 72000|Maria Isabel Cortez|
|    Chris|      NULL|   Evans|  35|      NULL|  Marketing| 60000|        Chris Evans|
|     Anna|     Marie|    NULL|  30|         2|      Sales|  NULL|         Anna Marie|
|     NULL|      NULL|   Lopez|  40|        10|         HR| 80000|              Lopez|
|    James|    Andrew|   Brown|NULL|         3|       NULL| 55000| James Andrew Brown|
|    Sarah|      NULL|  Miller|  27|         1|Engineering| 65000|       Sarah Miller|
+---------+----------+--------+----+----------+-----------+------+-------------------+



In [22]:
#OR 

df_concat = test_df.withColumn("FullName", F.concat_ws(" ", *[F.col(c) for c in ["firstName","middleName","lastName"]]))
df_concat.show()

+---------+----------+--------+----+----------+-----------+------+-------------------+
|firstName|middleName|lastName| Age|Experience| Department|Salary|           FullName|
+---------+----------+--------+----+----------+-----------+------+-------------------+
|     John|      NULL|   Smith|  28|         3|      Sales| 50000|         John Smith|
|    Maria|    Isabel|  Cortez|NULL|         5|Engineering| 72000|Maria Isabel Cortez|
|    Chris|      NULL|   Evans|  35|      NULL|  Marketing| 60000|        Chris Evans|
|     Anna|     Marie|    NULL|  30|         2|      Sales|  NULL|         Anna Marie|
|     NULL|      NULL|   Lopez|  40|        10|         HR| 80000|              Lopez|
|    James|    Andrew|   Brown|NULL|         3|       NULL| 55000| James Andrew Brown|
|    Sarah|      NULL|  Miller|  27|         1|Engineering| 65000|       Sarah Miller|
+---------+----------+--------+----+----------+-----------+------+-------------------+



In [21]:
df_dropna = test_df.dropna()
df_dropna.show()

+---------+----------+--------+---+----------+----------+------+
|firstName|middleName|lastName|Age|Experience|Department|Salary|
+---------+----------+--------+---+----------+----------+------+
+---------+----------+--------+---+----------+----------+------+



In [11]:
df_dropnaAll = test_df.dropna(how='all')
df_dropnaAll.show()

+---------+----------+--------+----+----------+-----------+------+
|firstName|middleName|lastName| Age|Experience| Department|Salary|
+---------+----------+--------+----+----------+-----------+------+
|     John|      NULL|   Smith|  28|         3|      Sales| 50000|
|    Maria|    Isabel|  Cortez|NULL|         5|Engineering| 72000|
|    Chris|      NULL|   Evans|  35|      NULL|  Marketing| 60000|
|     Anna|     Marie|    NULL|  30|         2|      Sales|  NULL|
|     NULL|      NULL|   Lopez|  40|        10|         HR| 80000|
|    James|    Andrew|   Brown|NULL|         3|       NULL| 55000|
|    Sarah|      NULL|  Miller|  27|         1|Engineering| 65000|
+---------+----------+--------+----+----------+-----------+------+



In [12]:
df_dropnaThresh = test_df.dropna(thresh=6)
df_dropnaThresh.show()

+---------+----------+--------+----+----------+-----------+------+
|firstName|middleName|lastName| Age|Experience| Department|Salary|
+---------+----------+--------+----+----------+-----------+------+
|     John|      NULL|   Smith|  28|         3|      Sales| 50000|
|    Maria|    Isabel|  Cortez|NULL|         5|Engineering| 72000|
|    Sarah|      NULL|  Miller|  27|         1|Engineering| 65000|
+---------+----------+--------+----+----------+-----------+------+



In [13]:
df_filteredDrop = test_df.dropna(subset=['middleName', 'Salary'])
df_filteredDrop.show()

+---------+----------+--------+----+----------+-----------+------+
|firstName|middleName|lastName| Age|Experience| Department|Salary|
+---------+----------+--------+----+----------+-----------+------+
|    Maria|    Isabel|  Cortez|NULL|         5|Engineering| 72000|
|    James|    Andrew|   Brown|NULL|         3|       NULL| 55000|
+---------+----------+--------+----+----------+-----------+------+



In [15]:
df_filteredRows = test_df.filter((test_df["Department"] == "Sales") & (test_df["Experience"] >= 3))
df_filteredRows.show() 

+---------+----------+--------+---+----------+----------+------+
|firstName|middleName|lastName|Age|Experience|Department|Salary|
+---------+----------+--------+---+----------+----------+------+
|     John|      NULL|   Smith| 28|         3|     Sales| 50000|
+---------+----------+--------+---+----------+----------+------+



In [19]:
from pyspark.ml.feature import Imputer


test_df_updated = test_df.fillna("N/A", subset=["middleName"])

imputer = Imputer(inputCols=["Age", "Salary"], outputCols=["Age_imputed", "Salary_imputed"]).setStrategy("mean")
imputer_model = imputer.fit(test_df_updated).transform(test_df_updated)
imputer_model.show()

+---------+----------+--------+----+----------+-----------+------+-----------+--------------+
|firstName|middleName|lastName| Age|Experience| Department|Salary|Age_imputed|Salary_imputed|
+---------+----------+--------+----+----------+-----------+------+-----------+--------------+
|     John|       N/A|   Smith|  28|         3|      Sales| 50000|         28|         50000|
|    Maria|    Isabel|  Cortez|NULL|         5|Engineering| 72000|         32|         72000|
|    Chris|       N/A|   Evans|  35|      NULL|  Marketing| 60000|         35|         60000|
|     Anna|     Marie|    NULL|  30|         2|      Sales|  NULL|         30|         63666|
|     NULL|       N/A|   Lopez|  40|        10|         HR| 80000|         40|         80000|
|    James|    Andrew|   Brown|NULL|         3|       NULL| 55000|         32|         55000|
|    Sarah|       N/A|  Miller|  27|         1|Engineering| 65000|         27|         65000|
+---------+----------+--------+----+----------+-----------+-